# MURE Automated Fine-Tuning (GitHub & Google Drive Integrated) 🚀
ဒီ Notebook က GitHub ကနေ Colab ကို ချိတ်ပြီး Google Drive က Dataset နဲ့ AI (Gemma-2) ကို Auto Fine-Tuning လုပ်ပေးမှာဖြစ်ပါတယ်။
ပြီးရင် Fine-tuned လုပ်ထားတဲ့ Model ကို Google Drive ထဲက `svo cc brain` folder အောက်မှာ အလိုအလျောက် ပြန်သိမ်းပေးပါမယ်။

**လုပ်ဆောင်ရမည့်အဆင့်များ:**
1. မိမိ Project ကို GitHub ပေါ်သို့ တင်ပါ။ (Upload to GitHub)
2. GitHub ပေါ်က `MURE_LLM_FINETUNE.ipynb` ကို ဖွင့်ပြီး **Open in Colab** ခလုတ်ကို နှိပ်ပါ။ (သို့မဟုတ် https://colab.research.google.com ကနေ GitHub Repo ကို ချိတ်ပြီး ဖွင့်ပါ)
3. AI Studio (MURE Web UI) ထဲက Settings -> `DATASET JSONL` ကို ဒေါင်းလုဒ်ဆွဲပါ။ (သို့မဟုတ် download လုပ်ထားသော `rules.json` ကို ယူပါ)
4. မိမိ Google Drive ထဲက `svo cc brain` ဆိုတဲ့ Folder ထဲသို့ ထိုဖိုင်ကို ထည့်ပါ။ 
5. Colab ရဲ့ အပေါ်က **Runtime -> Run All** ကို နှိပ်ပြီး Google Drive ကို Allow/Mount လုပ်ပေးလိုက်ပါ။

In [ ]:
# 1. Setup Unsloth and Dependencies
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json

# Path ကို သတ်မှတ်ခြင်း
drive_folder = "/content/drive/MyDrive/svo cc brain"
jsonl_path = os.path.join(drive_folder, "mure_finetune_dataset.jsonl")
fallback_json = os.path.join(drive_folder, "rules.json")
dataset_to_use = "dataset.jsonl"

if not os.path.exists(drive_folder):
    os.makedirs(drive_folder, exist_ok=True)
    print(f"⚠️ {drive_folder} ကို အသစ်တည်ဆောက်လိုက်ပါတယ်။ ထိုထဲသို့ dataset ဖိုင်ကို ထည့်ပေးပါ။")
    
if os.path.exists(jsonl_path):
    print(f"✅ {jsonl_path} ကို တွေ့ရှိပါတယ်။")
    !cp "{jsonl_path}" "{dataset_to_use}"
elif os.path.exists(fallback_json):
    print(f"✅ {fallback_json} (rules.json) ကို တွေ့ရှိပါတယ်။ JSONL အဖြစ် ပြောင်းနေပါသည်...")
    # Convert rules.json to jsonl format automatically
    try:
        with open(fallback_json, 'r', encoding='utf-8') as f:
            data = json.load(f)
            causal_memory = data.get('causalMemory', [])
        with open(dataset_to_use, 'w', encoding='utf-8') as out_f:
            for rule in causal_memory:
                if 'cause' in rule and 'effect' in rule:
                    cause = str(rule['cause']).strip()
                    effect = str(rule['effect']).strip()
                    strength = f" (Confidence: {rule['strength']})" if 'strength' in rule else ""
                    import random
                    type = random.randint(0, 3)
                    if type == 0:
                        instruction = f"What is the logical consequence of the following situation: \"{cause}\"?"
                        output = f"The logical consequence of \"{cause}\" is \"{effect}\"{strength}."
                    elif type == 1:
                        instruction = f"Analyze the causal relationship: What happens as a result of \"{cause}\"?"
                        output = f"Based on logical deduction, \"{cause}\" leads to \"{effect}\"{strength}."
                    elif type == 2:
                        instruction = f"Given the cause: \"{cause}\", determine the expected effect."
                        output = f"The expected effect is: \"{effect}\". This causal link is established by MURE reasoning."
                    else:
                        instruction = f"Predict the outcome if this condition is met: \"{cause}\""
                        output = f"If this condition is met, it will inevitably result in \"{effect}\"{strength}."
                    out_f.write(json.dumps({"instruction": instruction, "input": "", "output": output}) + '\\n')
        print("✅ Converted rules.json to dataset.jsonl")
    except Exception as e:
        print(f"Error converting json: {e}")
else:
    raise FileNotFoundError(f"⚠️ Dataset ဖိုင်ကို မတွေ့ပါ။ {drive_folder} အောက်တွင် mure_finetune_dataset.jsonl (သို့) rules.json ကို ထည့်ပေးပါ။")

In [ ]:
# 3. Load Model with Unsloth
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Model ကို Gemma-2 (2 Billion Parameters) နဲ့ အစမ်းသုံးပါမယ်။
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
# 4. Apply Formatting to Dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files=dataset_to_use, split="train")

alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples.get("input", [""] * len(instructions))
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

if len(dataset) == 0:
    raise ValueError("⚠️ Dataset empty. Please ensure valid rules exist in your jsonl or rules.json file.")

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
# 5. Train the Model
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # လိုအပ်ပါက epoch တိုးနိုင်ပါသည်
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Training Started...")
trainer_stats = trainer.train()
print("✅ Training Completed!")

In [ ]:
# 6. Save Model to Google Drive Automatically
output_dir = os.path.join(drive_folder, "MURE_Finetuned_Model")
print(f"💾 Saving Fine-tuned model to {output_dir} ...")

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ အောင်မြင်စွာ သိမ်းဆည်းပြီးပါပြီ! Google Drive ထဲတွင် 'svo cc brain/MURE_Finetuned_Model' အမည်ဖြင့် ဝင်ရောက်ကြည့်ရှုနိုင်ပါသည်။")

In [ ]:
# 7. Test the Fine-Tuned Model! (Inference)
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[alpaca_prompt.format("What is the logical consequence of the following situation: \"high temperatures\"?", "", "")], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print("\n--- MURE PREDICTION ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

# (Optional) Export to GGUF
# model.save_pretrained_gguf(os.path.join(output_dir, "gguf"), tokenizer, quantization_method="q4_k_m")
print("✅ Done!")